In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar el dataset
df = pd.read_csv("Clean_Dataset.csv", index_col=0)

print("📊 --- RESPUESTAS A LAS PREGUNTAS DE INVESTIGACIÓN --- 📊\n")

# a) ¿Varía el precio según las aerolíneas?
print("a) Precio promedio por Aerolínea:")
print(df.groupby('airline')['price'].mean().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))
print("-" * 50)

# b) ¿Cómo se ve afectado el precio cuando se compra 1 o 2 días antes?
print("b) Impacto de la anticipación (Días restantes vs Precio Promedio):")
print(df[df['days_left'] <= 5].groupby('days_left')['price'].mean().apply(lambda x: f"${x:,.2f}"))
print("Insight: Comprar con 1 día de anticipación dispara drásticamente el costo promedio.")
print("-" * 50)

# c) ¿El precio cambia según la hora de salida?
print("c) Precio promedio según Horario de Salida:")
print(df.groupby('departure_time')['price'].mean().sort_values(ascending=False).apply(lambda x: f"${x:,.2f}"))
print("-" * 50)

# d) ¿Cómo cambia el precio con el origen y destino?
print("d) Top 5 Rutas más caras (Origen -> Destino):")
rutas = df.groupby(['source_city', 'destination_city'])['price'].mean().sort_values(ascending=False).head(5)
print(rutas.apply(lambda x: f"${x:,.2f}"))
print("-" * 50)

# e) ¿Cómo varía entre clase Económica y Business?
print("e) Brecha de precio por Clase de Asiento:")
print(df.groupby('class')['price'].mean().apply(lambda x: f"${x:,.2f}"))

📊 --- RESPUESTAS A LAS PREGUNTAS DE INVESTIGACIÓN --- 📊

a) Precio promedio por Aerolínea:
airline
Vistara      $7,764.71
Air_India    $7,281.61
SpiceJet     $6,252.01
GO_FIRST     $5,643.95
Indigo       $5,411.14
AirAsia      $4,140.54
Name: price, dtype: object
--------------------------------------------------
b) Impacto de la anticipación (Días restantes vs Precio Promedio):
days_left
1.0    $14,565.51
2.0    $13,710.47
3.0    $13,015.23
4.0    $10,783.74
5.0    $10,540.41
Name: price, dtype: object
Insight: Comprar con 1 día de anticipación dispara drásticamente el costo promedio.
--------------------------------------------------
c) Precio promedio según Horario de Salida:
departure_time
Morning          $7,130.39
Afternoon        $6,505.77
Early_Morning    $6,496.59
Evening          $6,385.74
Night            $6,151.97
Late_Night       $4,751.23
Name: price, dtype: object
--------------------------------------------------
d) Top 5 Rutas más caras (Origen -> Destino):
source_city

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Eliminar filas con valores nulos
df.dropna(inplace=True)

# 1. Definir variables predictoras (X) y objetivo (y)
# Quitamos 'flight' porque es un identificador de código único con demasiadas categorías
X = df.drop(columns=['price', 'flight'])
y = df['price']

# Mapeo manual de variables con orden jerárquico implícito (Ordinal Encoding manual)
X['class'] = X['class'].map({'Economy': 0, 'Business': 1})
X['stops'] = X['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})

# Identificar columnas para One-Hot Encoding automático (las que siguen siendo texto)
categorical_features = ['airline', 'source_city', 'departure_time', 'arrival_time', 'destination_city']
numeric_features = ['duration', 'days_left', 'class', 'stops']

# 2. Configurar el Preprocesador Estadístico
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), # Normaliza escalas numéricas y distancias
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features) # Texto a binario
    ])

# 3. División de datos en Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Definición de los Modelos Solicitados
modelos = {
    "Regresión Lineal Baseline": LinearRegression(),
    "K-Neighbors Regressor (KNN)": KNeighborsRegressor(n_neighbors=5, weights='distance'),
    "XGBoost Regressor (Conjunto)": XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
}

print("\n🚀 --- ENTRENANDO Y EVALUANDO MODELOS --- 🚀\n")

# Evaluar cada modelo secuencialmente
for nombre, algoritmo in modelos.items():
    # Estructurar el pipeline de producción
    pipeline_completo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', algoritmo)
    ])

    # Entrenar el modelo
    pipeline_completo.fit(X_train, y_train)

    # Generar predicciones en el set de testeo (el examen)
    y_pred = pipeline_completo.fit(X_train, y_train).predict(X_test)

    # Calcular Métricas
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Mostrar resultados ejecutivos
    print(f"📌 Modelo: {nombre}")
    print(f"   • MAE (Error Absoluto Medio): ${mae:,.2f}")
    print(f"   • RMSE (Castigo a desvíos):    ${rmse:,.2f}")
    print(f"   • R² Score (Precisión):        {r2:.4f}")
    print("-" * 60)


🚀 --- ENTRENANDO Y EVALUANDO MODELOS --- 🚀

📌 Modelo: Regresión Lineal Baseline
   • MAE (Error Absoluto Medio): $1,870.85
   • RMSE (Castigo a desvíos):    $2,572.77
   • R² Score (Precisión):        0.5067
------------------------------------------------------------
📌 Modelo: K-Neighbors Regressor (KNN)
   • MAE (Error Absoluto Medio): $843.41
   • RMSE (Castigo a desvíos):    $1,635.50
   • R² Score (Precisión):        0.8007
------------------------------------------------------------
📌 Modelo: XGBoost Regressor (Conjunto)
   • MAE (Error Absoluto Medio): $1,060.09
   • RMSE (Castigo a desvíos):    $1,699.21
   • R² Score (Precisión):        0.7848
------------------------------------------------------------
